Import Libraries

In [ ]:
import numpy as np
import torch

import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import torch.nn as nn
from torchsummary import summary

from torchvision import transforms, datasets
from torch.utils.data import DataLoader

Load data

In [ ]:
train_gen = ImageDataGenerator()

train_ds = train_gen.flow_from_directory(
    '/content/drive/MyDrive/Colab Notebooks/LEARNING/REPO/data/train',
    target_size=(200, 200),
    batch_size=32
)

test_gen = ImageDataGenerator()

test_ds = test_gen.flow_from_directory(
    '/content/drive/MyDrive/Colab Notebooks/LEARNING/REPO/data/test',
    target_size=(200, 200),
    batch_size=32
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Found 800 images belonging to 2 classes.
Found 201 images belonging to 2 classes.


In [ ]:
class HairClassifier(nn.Module):
    def __init__(self):
        super(HairClassifier, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
model = HairClassifier()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

Using device: cpu


In [ ]:
criterion = criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

Q1 (Which loss function you will use?) :

In [ ]:
# Using BCEWithLogitsLoss is preferred because it merges the sigmoid activation
# and binary cross-entropy into one operation, giving better numerical stability.


Q2 (What's the total number of parameters of the model? You can use torchsummary or count manually)

In [ ]:
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
              ReLU-2         [-1, 32, 198, 198]               0
         MaxPool2d-3           [-1, 32, 99, 99]               0
            Linear-4                   [-1, 64]      20,072,512
              ReLU-5                   [-1, 64]               0
            Linear-6                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 21.54
Params size (MB): 76.57
Estimated Total Size (MB): 98.57
----------------------------------------------------------------


Model Training

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = '/content/drive/MyDrive/Colab Notebooks/LEARNING/REPO/data/test'
train_data = '/content/drive/MyDrive/Colab Notebooks/LEARNING/REPO/data/train'

train_dataset = datasets.ImageFolder(train_data, transform=train_transforms)
test_dataset = datasets.ImageFolder(test_data, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=20, shuffle=False)

len(train_dataset), len(test_dataset)


(800, 201)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(test_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6665, Acc: 0.6112, Val Loss: 0.6511, Val Acc: 0.6617
Epoch 2/10, Loss: 0.5702, Acc: 0.6787, Val Loss: 0.6332, Val Acc: 0.6318
Epoch 3/10, Loss: 0.5207, Acc: 0.7350, Val Loss: 0.6143, Val Acc: 0.6766
Epoch 4/10, Loss: 0.4773, Acc: 0.7600, Val Loss: 0.6049, Val Acc: 0.6617
Epoch 5/10, Loss: 0.4606, Acc: 0.7550, Val Loss: 0.7307, Val Acc: 0.5672
Epoch 6/10, Loss: 0.3954, Acc: 0.8275, Val Loss: 0.6412, Val Acc: 0.6866
Epoch 7/10, Loss: 0.2844, Acc: 0.8838, Val Loss: 0.8307, Val Acc: 0.6816
Epoch 8/10, Loss: 0.2885, Acc: 0.8788, Val Loss: 0.7052, Val Acc: 0.7114
Epoch 9/10, Loss: 0.1882, Acc: 0.9313, Val Loss: 0.9275, Val Acc: 0.6866
Epoch 10/10, Loss: 0.2585, Acc: 0.8912, Val Loss: 0.8158, Val Acc: 0.6915


Q3 (What is the median of training accuracy for all the epochs for this model?)

In [ ]:
median_train_acc = np.median(history['acc'])
median_train_acc

np.float64(0.79375)

Q4 (What is the standard deviation of training loss for all the epochs for this model?)

In [ ]:
std_train_loss = np.std(history['loss'])
std_train_loss

np.float64(0.146177316193373)

In [ ]:
aug_transforms = transforms.Compose([
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset_aug = datasets.ImageFolder(train_data, transform=aug_transforms)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=20, shuffle=True)

In [ ]:
n_epochs_aug = 10

aug_history = {'val_loss': [], 'val_acc': []}

for epoch in range(n_epochs_aug):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

    val_epoch_loss = val_running_loss / len(test_dataset)
    val_epoch_acc = correct_val / total_val

    aug_history['val_loss'].append(val_epoch_loss)
    aug_history['val_acc'].append(val_epoch_acc)

    print(f"[Augmented] Epoch {epoch+1}/{n_epochs_aug}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

[Augmented] Epoch 1/10, Val Loss: 0.6583, Val Acc: 0.6667
[Augmented] Epoch 2/10, Val Loss: 0.6539, Val Acc: 0.6766
[Augmented] Epoch 3/10, Val Loss: 0.6508, Val Acc: 0.6965
[Augmented] Epoch 4/10, Val Loss: 0.5643, Val Acc: 0.7065
[Augmented] Epoch 5/10, Val Loss: 0.5530, Val Acc: 0.7214
[Augmented] Epoch 6/10, Val Loss: 0.5923, Val Acc: 0.6816
[Augmented] Epoch 7/10, Val Loss: 0.6176, Val Acc: 0.6816
[Augmented] Epoch 8/10, Val Loss: 0.6749, Val Acc: 0.6418
[Augmented] Epoch 9/10, Val Loss: 0.6744, Val Acc: 0.6716
[Augmented] Epoch 10/10, Val Loss: 0.5490, Val Acc: 0.7114


Q5 (What is the mean of test loss for all the epochs for the model trained with augmentations?)

In [ ]:
mean_test_loss = np.mean(aug_history['val_loss'])
mean_test_loss

np.float64(0.6188491473743571)

Q6 (What's the average of test accuracy for the last 5 epochs (from 6 to 10) for the model trained with augmentations?)

In [ ]:
last_5_acc = aug_history['val_acc'][5:]
mean_last_5_acc = np.mean(last_5_acc)
mean_last_5_acc

np.float64(0.6776119402985075)